# M5 — thermal emulator (Marswalk)

Trains a small MLP surrogate of the surface-temperature model `physics.thermal.surface_temp_range_k` — fast (T_min, T_max) from (thermal inertia, insolation, albedo). The underlying model is a C3 **placeholder**; the emulator's realism is bounded by it — swap in a validated thermal model and retrain for production.

**How to run:** **Run all** (CPU is fine), then send back **`thermal_emulator_report.json`** and **`thermal_emulator_samples.png`**.

In [ ]:
!pip install -q "git+https://github.com/MONSTER4REX/Martian-Map.git"
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', dev)

## Build data from the physics model, train, evaluate on a fresh draw

In [ ]:
import json, time
import numpy as np
from marswalk.ml.thermal_emulator import build_dataset, train_mlp, evaluate

Xtr, Ytr = build_dataset(4000, seed=0)
Xte, Yte = build_dataset(1000, seed=99)
t0 = time.time()
model = train_mlp(Xtr, Ytr, epochs=200, device=dev)
rep = evaluate(model, Xte, Yte)
secs = time.time() - t0
print(rep)

In [ ]:
report = {**rep.as_dict(), 'device': (torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU'),
          'epochs': 200, 'seconds': round(secs, 1), 'final_loss': round(model.history[-1], 6)}
with open('thermal_emulator_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print(json.dumps(report, indent=2))

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pred = model.predict(Xte)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for j, name in enumerate(['T_min', 'T_max']):
    ax[j].scatter(Yte[:, j], pred[:, j], s=5, alpha=.3, color='#e2572b')
    lo, hi = Yte[:, j].min(), Yte[:, j].max()
    ax[j].plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax[j].set_xlabel(f'physics {name} (K)'); ax[j].set_ylabel(f'emulated {name} (K)')
    ax[j].set_title(name)
fig.suptitle(f"thermal emulator — MAE T_max {rep.mae_tmax_k:.2f} K, T_min {rep.mae_tmin_k:.2f} K")
fig.tight_layout(); fig.savefig('thermal_emulator_samples.png', dpi=120)
import os; print('SAVED (download these):', os.path.abspath('thermal_emulator_samples.png'), '|', os.path.abspath('thermal_emulator_report.json'))

## Send back

Download **`thermal_emulator_report.json`** and **`thermal_emulator_samples.png`** and send them to the lead. Thanks!